# 01 AI Agent 工程师导论

本章目标：明确 AI Agent 工程师的技能结构、Agent 系统组成、学习顺序，并用一个最小 planner 理解控制面。

## 定义

AI Agent 工程师负责把 LLM 接入业务系统，建设可运行、可观测、可评测、可控的智能应用。

工作对象：

1. LLM
2. 工具调用
3. 知识库
4. 对话上下文
5. 任务状态
6. 执行流程
7. 权限控制
8. 效果评测
9. 运行监控

## 核心技能

AI Agent 工程师需要具备以下技能。

1. 后端工程能力：Python / TypeScript、HTTP API、数据库、异步任务、测试、日志、Docker、部署。
2. LLM 应用能力：token、上下文窗口、采样参数、system prompt、structured output、function calling。
3. 工具封装能力：把业务 API、数据库、文件系统、第三方服务封装成模型可调用工具。
4. Schema 设计能力：设计工具参数、返回值、错误结构、字段约束、类型校验。
5. RAG 工程能力：文档清洗、chunk、embedding、向量检索、metadata、rerank、引用来源、召回评测。
6. 上下文管理能力：长对话、短期上下文、长期记忆、摘要压缩、任务状态恢复。
7. Agent Loop 设计能力：设计 plan、act、observe、retry、stop、handoff、human review 等执行流程。
8. 多 Agent 协作能力：任务拆分、角色分工、消息传递、冲突处理、结果汇总。
9. 评测能力：构建 eval dataset，评估回答质量、工具调用质量、任务完成率、失败类型。
10. 可观测性能力：记录 prompt、model response、tool call、latency、cost、trace id、error。
11. 安全控制能力：防 prompt injection、权限越权、数据泄露、危险工具误调用。
12. 人审流程设计能力：对删除、发送、付款、审批、医疗、法律等高风险动作加入人工确认。

## Agent 系统组成

Agent = LLM + Tools + Knowledge + State + Control Loop + Guardrails + Observability + Evals。

| 模块 | 职责 |
| --- | --- |
| LLM | 理解任务、生成内容、选择动作 |
| Tools | 调 API、查数据库、读写文件、执行业务动作 |
| Knowledge | 提供文档、规则、历史记录、外部数据 |
| State | 保存对话、任务进度、用户偏好、工具结果 |
| Control Loop | 控制执行步骤、重试、停止、转人工 |
| Guardrails | 限制输入、输出、权限、工具调用 |
| Observability | 记录链路、成本、延迟、错误、工具轨迹 |
| Evals | 用测试集衡量效果和回归 |

最小执行流程：

```text
用户目标
-> 解析任务
-> 获取上下文
-> 选择动作
-> 调用工具或生成回复
-> 观察结果
-> 更新状态
-> 继续执行或结束
```

## 常见系统类型

| 类型 | 主要能力 | 典型用途 |
| --- | --- | --- |
| Chatbot | 对话生成 | FAQ、陪聊、简单问答 |
| RAG | 检索增强回答 | 企业知识库、文档问答、政策查询 |
| Workflow | 固定流程自动化 | 审批、工单流转、数据同步 |
| Agent | 动态决策、工具调用、状态管理 | 研发周报助手、售后处理、资料整理、复杂任务执行 |

Agent 项目通常组合使用 RAG 和 Workflow。

## 业务案例

研发周报助手：

```text
读取代码提交和任务记录
-> 按项目和成员分类
-> 提取已完成事项
-> 提取风险和阻塞项
-> 生成周报草稿
-> 请求负责人确认
-> 发布到团队空间
-> 记录生成与发布过程
```

涉及能力：文件解析、任务系统查询、信息抽取、工具调用、状态流转、人工确认和审计日志。

## 学习顺序

1. LLM 基础：token、Transformer、Attention、生成机制、上下文窗口、幻觉。
2. Prompt 与结构化输出：system prompt、few-shot、JSON Schema、输出校验、失败样例。
3. Tool Calling：工具定义、参数 schema、调用流程、错误处理、权限控制。
4. MCP：host、client、server、tools、resources、prompts。
5. RAG：数据清洗、chunk、embedding、检索、rerank、引用、评测。
6. Context / Memory / State：长对话、记忆、摘要压缩、任务状态、恢复机制。
7. Agent Loop：plan-act-observe、ReAct、状态机、重试、人审。
8. Agent Framework：LangGraph、OpenAI Agents SDK、AutoGen、Microsoft Agent Framework、Dify、Coze。
9. Production：evals、tracing、guardrails、权限、成本、延迟、部署。
10. Capstone：做一个完整业务项目，覆盖知识库、工具、状态、人审、评测、监控。

## Minimal Planner

目标：用最小代码理解 Agent 控制面。

本节不调用 LLM，只演示以下判断：

1. 任务需要知识库
2. 任务需要工具调用
3. 任务需要人工审批
4. 任务风险等级
5. 下一步动作

### 1. 定义计划结构

Agent 执行前需要形成一个 plan。这里保留 5 个字段。

In [1]:
from dataclasses import asdict, dataclass
import json


@dataclass(frozen=True)
class Plan:
    task: str
    route: str
    needs: list[str]
    risk_level: str
    next_action: str


### 2. 配置任务识别规则

规则版本用于学习控制流程。后续章节会逐步替换为 LLM 判断、tool calling、RAG 和 Agent loop。

In [2]:
KNOWLEDGE_KEYWORDS = (
    "总结", "查询", "资料", "材料", "知识库", "文档", "纪要", "README", "病例", "合同"
)

TOOL_KEYWORDS = (
    "读取", "计算", "创建", "发送", "写入", "更新", "删除", "开平方", "上传", "下载"
)

HUMAN_REVIEW_KEYWORDS = (
    "删除", "发送", "付款", "审批", "发布", "诊断", "医疗", "合同", "转账"
)


def contains_any(text: str, keywords: tuple[str, ...]) -> bool:
    return any(keyword.lower() in text.lower() for keyword in keywords)


### 3. 实现 planner

输入：用户任务。

输出：任务路线、所需能力、风险等级、下一步动作。

In [3]:
def make_plan(task: str) -> Plan:
    needs: list[str] = []

    if contains_any(task, KNOWLEDGE_KEYWORDS):
        needs.append("knowledge_retrieval")

    if contains_any(task, TOOL_KEYWORDS):
        needs.append("tool_calling")

    if contains_any(task, HUMAN_REVIEW_KEYWORDS):
        needs.append("human_review")

    if not needs:
        needs.append("direct_answer")

    if "human_review" in needs:
        risk_level = "high"
        next_action = "Prepare a draft plan and wait for human approval before taking action."
    elif "tool_calling" in needs:
        risk_level = "medium"
        next_action = "Validate tool parameters, call the required tool, then summarize the result."
    elif "knowledge_retrieval" in needs:
        risk_level = "low"
        next_action = "Retrieve relevant context, cite sources, then answer."
    else:
        risk_level = "low"
        next_action = "Answer directly."

    route = "agentic_workflow" if len(needs) > 1 else needs[0]

    return Plan(
        task=task,
        route=route,
        needs=needs,
        risk_level=risk_level,
        next_action=next_action,
    )


### 4. 运行样例

观察不同任务触发的能力。

In [4]:
tasks = [
    "解释一下 AI Agent 是什么",
    "总结 README.md 的主要内容",
    "计算 16 的开平方",
    "读取会议纪要并发送团队通知",
]

for task in tasks:
    plan = make_plan(task)
    print(json.dumps(asdict(plan), ensure_ascii=False, indent=2))
    print("-" * 60)


{
  "task": "解释一下 AI Agent 是什么",
  "route": "direct_answer",
  "needs": [
    "direct_answer"
  ],
  "risk_level": "low",
  "next_action": "Answer directly."
}
------------------------------------------------------------
{
  "task": "总结 README.md 的主要内容",
  "route": "knowledge_retrieval",
  "needs": [
    "knowledge_retrieval"
  ],
  "risk_level": "low",
  "next_action": "Retrieve relevant context, cite sources, then answer."
}
------------------------------------------------------------
{
  "task": "计算 16 的开平方",
  "route": "tool_calling",
  "needs": [
    "tool_calling"
  ],
  "risk_level": "medium",
  "next_action": "Validate tool parameters, call the required tool, then summarize the result."
}
------------------------------------------------------------
{
  "task": "读取会议纪要并发送团队通知",
  "route": "agentic_workflow",
  "needs": [
    "knowledge_retrieval",
    "tool_calling",
    "human_review"
  ],
  "risk_level": "high",
  "next_action": "Prepare a draft plan and wait for human approva

### 5. 验证关键行为

学习阶段也要保留断言。断言用于固定预期行为。

In [5]:
direct_plan = make_plan("解释一下 AI Agent 是什么")
assert direct_plan.needs == ["direct_answer"]
assert direct_plan.risk_level == "low"

tool_plan = make_plan("计算 16 的开平方")
assert "tool_calling" in tool_plan.needs
assert tool_plan.risk_level == "medium"

review_plan = make_plan("读取会议纪要并发送团队通知")
assert "knowledge_retrieval" in review_plan.needs
assert "tool_calling" in review_plan.needs
assert "human_review" in review_plan.needs
assert review_plan.risk_level == "high"

print("All checks passed.")


All checks passed.


### 6. 当前版本的边界

1. 规则依赖关键词，泛化能力弱。
2. 同义词、语义隐含、复杂任务拆解处理不好。
3. 没有真正调用工具。
4. 没有外部知识检索。
5. 没有状态持久化。

后续章节逐步替换这些部分。

## 上线检查项

1. 工具权限：每个工具有明确权限、参数校验、错误返回。
2. 数据权限：RAG 检索支持 metadata 过滤和用户权限过滤。
3. 状态管理：对话状态、任务状态、工具结果分层存储。
4. Trace 记录：每次模型调用和工具调用可追踪。
5. Eval 集合：有固定测试集，支持版本回归。
6. 失败处理：支持重试、降级、终止、转人工。
7. 人工审批：高风险动作执行前进入人审。
8. 成本控制：记录 token、模型、延迟、调用次数。
9. 安全防护：处理 prompt injection、越权访问、敏感信息泄露。
10. 审计日志：记录关键输入、输出、决策和执行结果。

## 常见误区

1. 只做 prompt，不做工具、状态、评测。
2. 只看最终回答，不看中间执行轨迹。
3. 只做 demo，不设计失败处理。
4. 只接 RAG，不做召回评测。
5. 只加多 Agent，不定义协作协议。
6. 只用框架，不理解底层执行流程。
7. 高风险工具缺少权限和人审。
8. 没有 eval dataset，靠主观体验判断效果。
9. 没有 tracing，线上失败无法定位。

## 本章要点

1. AI Agent 工程师的核心能力是 LLM 应用工程化。
2. Agent 系统由模型、工具、知识、状态、控制、安全、观测、评测组成。
3. 学习路线优先覆盖工程能力，再选择框架。
4. 生产级 Agent 必须具备权限、评测、追踪、人审和审计。